# Seed replication -- 2x2 factorial (state x spill-penalty)

## 1. Imports & constants

In [ ]:
from pathlib import Path
import json, gzip, re, pickle, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import csr_matrix, vstack as sp_vstack
from scipy.stats import pearsonr

import gymnasium as gym
from gymnasium import spaces

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

ED_LEVELS = [round(x * 0.05, 2) for x in range(1, 21)]


## 2. Data loading

In [ ]:
def _first_existing(folder, names, required=True):
    folder = Path(folder)
    for n in names:
        if (folder / n).exists(): return folder / n
    if required: raise FileNotFoundError(f"None of {names} in {folder}")
    return None

def _read_json(path):
    path = Path(path)
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f: return json.load(f)

def _norm_col(df, name):
    df = df.copy()
    if len(df.columns) > 0 and str(df.columns[0]).lower().startswith("unnamed"):
        df = df.rename(columns={df.columns[0]: name})
    return df

def _load_profiles_and_ic(folder):
    folder = Path(folder)
    if (folder / "profiles.csv").exists():
        return (_norm_col(pd.read_csv(folder/"profiles.csv"),"hour"),
                _norm_col(pd.read_csv(folder/"initial_conditions.csv"),"generator"))
    xls = pd.ExcelFile(folder/"explanatory_variables.xlsx")
    def find(kws):
        for k in kws:
            for s in xls.sheet_names:
                if all(x.lower() in s.lower() for x in k): return s
        return xls.sheet_names[0]
    return (_norm_col(pd.read_excel(folder/"explanatory_variables.xlsx",
                                    sheet_name=find([("profile",)])),"hour"),
            _norm_col(pd.read_excel(folder/"explanatory_variables.xlsx",
                                    sheet_name=find([("initial",)])),"generator"))

def get_profile_series(df):
    def fc(kws):
        for c in df.columns:
            if all(k in str(c).lower() for k in kws): return c
        return None
    d = pd.to_numeric(df[fc(["demand"])],errors="coerce").fillna(0).to_numpy(float)
    w = pd.to_numeric(df[fc(["wind"])],errors="coerce").fillna(0).to_numpy(float) if fc(["wind"]) else np.zeros_like(d)
    s = pd.to_numeric(df[fc(["solar"])],errors="coerce").fillna(0).to_numpy(float) if fc(["solar"]) else np.zeros_like(d)
    return d, w, s, d-w-s

def list_instance_dirs(root):
    dirs = sorted(p for p in Path(root).rglob("instance_*") if p.is_dir())
    if not dirs: raise FileNotFoundError(f"No instance_* under {root}")
    return dirs

def split_instance_dirs(dirs, val_quarters=None):
    val_quarters = val_quarters or set()
    train, valid = [], []
    for d in dirs:
        m = re.match(r"instance_(\d{4})_(Q[1-4])_(\d+)", d.name)
        if m and (int(m.group(1)), m.group(2)) in val_quarters: valid.append(d)
        else: train.append(d)
    return train, valid

def load_single_bundle(d):
    d = Path(d)
    profiles, ic = _load_profiles_and_ic(d)
    inp = _read_json(_first_existing(d,["InputData.json","InputData.json.gz"]))
    outp_path = _first_existing(d,["OutputData.json","OutputData.json.gz"],required=False)
    outp = _read_json(outp_path) if outp_path else None
    return {"instance_id":d.name,"profiles":profiles,"initial_conditions":ic,
            "input_json":inp,"output_json":outp}

def _as_float_array(v):
    if v is None: return np.array([], float)
    return pd.to_numeric(pd.Series(v if isinstance(v,list) else [v]),
                          errors="coerce").dropna().to_numpy(float)

def approx_marginal_cost(mw, cost):
    mw, cost = _as_float_array(mw), _as_float_array(cost)
    if len(mw) < 2: return 0.0
    return float((cost[1]-cost[0])/max(mw[1]-mw[0],1e-9))

def parse_generators(inp):
    gen = pd.DataFrame.from_dict(inp["Generators"],orient="index").reset_index().rename(columns={"index":"generator"})
    gen["Type_clean"] = gen["Type"].str.strip().str.lower()
    th = gen[gen["Type_clean"]=="thermal"].copy()
    if th.empty: raise ValueError("No thermal generators")
    th["cost_curve_mw"] = th["Production cost curve (MW)"].apply(_as_float_array)
    th["cost_curve_usd"] = th["Production cost curve ($)"].apply(_as_float_array)
    th["pmin"] = th["cost_curve_mw"].apply(lambda x: float(x[0]) if len(x) else 0.0)
    th["pmax"] = th["cost_curve_mw"].apply(lambda x: float(x[-1]) if len(x) else 0.0)
    th["marginal_cost"] = th.apply(lambda r: approx_marginal_cost(r["cost_curve_mw"],r["cost_curve_usd"]),axis=1)
    th["startup_cost"] = th["Startup costs ($)"].apply(
        lambda x: float(_as_float_array(x)[0]) if len(_as_float_array(x)) else 0.0)
    for src,tgt in [("Initial power (MW)","init_power"),("Initial status (h)","init_status"),
                    ("Minimum uptime (h)","min_up"),("Minimum downtime (h)","min_down"),
                    ("Ramp up limit (MW)","ramp_up"),("Ramp down limit (MW)","ramp_down")]:
        th[tgt] = pd.to_numeric(th.get(src,0),errors="coerce").fillna(0.0)
    th = th.sort_values(["marginal_cost","startup_cost","pmax"]).reset_index(drop=True)
    return th[["generator","cost_curve_mw","cost_curve_usd","pmin","pmax",
               "marginal_cost","startup_cost","init_power","init_status",
               "min_up","min_down","ramp_up","ramp_down"]]

def get_balance_penalty(inp, default=12533.0):
    params = inp.get("Parameters",{})
    for k in ["Power balance penalty ($/MW)","Power balance penalty"]:
        if k in params:
            v = pd.to_numeric(pd.Series([params[k]]),errors="coerce").iloc[0]
            if pd.notna(v): return float(v)
    return default

def bundle_to_episode(b):
    th = parse_generators(b["input_json"])
    d, w, s, nd = get_profile_series(b["profiles"])
    return {"instance_id":b["instance_id"],"thermal_df":th,
            "demand":d,"wind":w,"solar":s,"net_demand":nd,
            "output_json":b["output_json"],
            "balance_penalty":get_balance_penalty(b["input_json"])}

def validate_compat(eps):
    ref = eps[0]["thermal_df"]["generator"].tolist()
    for e in eps[1:]:
        if e["thermal_df"]["generator"].tolist() != ref:
            raise ValueError(f"Mismatch: {e['instance_id']}")
    return True

def load_episodes(dirs, max_n=None, seed=42, verbose=True):
    dirs = list(dirs)
    np.random.default_rng(seed).shuffle(dirs)
    if max_n: dirs = dirs[:max_n]
    eps, fails = [], []
    for d in dirs:
        try: eps.append(bundle_to_episode(load_single_bundle(d)))
        except Exception as e: fails.append((str(d),repr(e)))
    if not eps: raise RuntimeError(f"No episodes. Failures: {fails[:3]}")
    validate_compat(eps)
    if verbose:
        print(f"Loaded {len(eps)} | Failed {len(fails)} | "
              f"Generators: {len(eps[0]['thermal_df'])} | Horizon: {len(eps[0]['demand'])}h")
    return eps


In [ ]:
DATA_ROOT  = Path(__import__("os").environ.get("UC_DATA_ROOT", "../rl_ready_competition_data"))
TRAIN_ROOT = DATA_ROOT / "train"
N_TRAIN, N_VALID, N_TEST = 300, 25, 100

all_dirs = list_instance_dirs(TRAIN_ROOT)
train_dirs, valid_dirs = split_instance_dirs(all_dirs, {(2026,"Q4"),(2027,"Q4"),(2028,"Q4"),(2029,"Q4")})
train_episodes = load_episodes(train_dirs, max_n=N_TRAIN, seed=42)
valid_episodes = load_episodes(valid_dirs, max_n=N_VALID, seed=43)

used_ids  = {e["instance_id"] for e in train_episodes} | {e["instance_id"] for e in valid_episodes}
test_dirs = [d for d in all_dirs if d.name not in used_ids]
test_episodes = load_episodes(test_dirs, max_n=N_TEST, seed=99)

assert len(test_episodes) == 100, f"Expected 100 test instances, got {len(test_episodes)}"


## 3. Environment (`UCOnlyEnv`)

In [ ]:
class UCOnlyEnv:
    def __init__(self, episodes, spill_penalty=8000.0, reward_scale=1e5,
                 random_reset=True, seed=42, lookahead=8,
                 state_version="v2", obs_window=72):
        validate_compat(episodes)
        self.episodes = list(episodes)
        self.spill_penalty = spill_penalty
        self.reward_scale = reward_scale
        self.random_reset = random_reset
        self.rng = np.random.default_rng(seed)
        self.lookahead = lookahead
        self.state_version = state_version
        self.obs_window = int(obs_window)

        th = episodes[0]["thermal_df"].reset_index(drop=True)
        self.generator_names = th["generator"].tolist()
        self.n_gen = len(th)
        self.horizon = len(episodes[0]["demand"])
        self.pmin = th["pmin"].to_numpy(float)
        self.pmax = th["pmax"].to_numpy(float)
        self.marginal_cost = th["marginal_cost"].to_numpy(float)
        self.startup_cost_arr = th["startup_cost"].to_numpy(float)
        self.min_up = th["min_up"].to_numpy(float)
        self.min_down = th["min_down"].to_numpy(float)
        self.ramp_up = th["ramp_up"].to_numpy(float)
        self.ramp_down = th["ramp_down"].to_numpy(float)
        self.cost_curve_mw = th["cost_curve_mw"].tolist()
        self.cost_curve_usd = th["cost_curve_usd"].tolist()
        self.thermal_cap = float(self.pmax.sum())

        self._mu_scale = max(float(self.min_up.max()), 1.0)
        self._md_scale = max(float(self.min_down.max()), 1.0)

        if self.state_version == "v1":
            self.obs_dim = 1 + self.n_gen + self.horizon + self.n_gen + 4
        else:
            self.obs_dim = 3 + self.obs_window + 4 + 3 * self.n_gen

        self.reset(options={"episode_index": 0})

    @property
    def obs_layout(self):
        n, W = self.n_gen, self.obs_window
        if self.state_version == "v1":
            H = self.horizon
            return {"time": (0, 1), "prev_on": (1, 1 + n),
                    "net_demand": (1 + n, 1 + n + H),
                    "status": (1 + n + H, 1 + n + H + n),
                    "lookahead": (1 + n + H + n, 1 + n + H + n + 4)}
        o = 0
        lay = {}
        lay["time"] = (o, o + 3); o += 3
        lay["window"] = (o, o + W); o += W
        lay["lookahead"] = (o, o + 4); o += 4
        lay["on_off"] = (o, o + n); o += n
        lay["can_start"] = (o, o + n); o += n
        lay["can_stop"] = (o, o + n); o += n
        return lay

    def reset(self, seed=None, options=None):
        if seed: self.rng = np.random.default_rng(seed)
        options = options or {}
        idx = int(options.get("episode_index",
                  self.rng.integers(len(self.episodes)) if self.random_reset else 0))
        self.ep_idx = idx
        ep = self.episodes[idx]
        self.demand = np.asarray(ep["demand"], float)
        self.wind   = np.asarray(ep["wind"], float)
        self.solar  = np.asarray(ep["solar"], float)
        self.net_demand = self.demand - self.wind - self.solar
        self.balance_penalty = float(ep.get("balance_penalty", 12533.0))

        th = ep["thermal_df"]
        self.current_dispatch = th["init_power"].to_numpy(float).copy()
        self.status_duration  = th["init_status"].to_numpy(float).copy()
        self.prev_on = (self.current_dispatch > 1e-6).astype(int)
        self.t = 0
        return self._obs(), {"instance_id": ep["instance_id"]}

    def _constraint_clocks(self):
        sd = self.status_duration
        is_on = (self.current_dispatch > 1e-6) | (sd > 0)
        hours_off = np.where(sd < 0, -sd, 0.0)
        can_start = np.where(is_on, 0.0, np.maximum(0.0, self.min_down - hours_off))
        hours_on = np.where(sd > 0, sd, 0.0)
        can_stop = np.where(~is_on, 0.0, np.maximum(0.0, self.min_up - hours_on))
        return (can_start / self._md_scale, can_stop / self._mu_scale, is_on.astype(float))

    def _demand_window(self):
        cap = max(self.thermal_cap, 1.0)
        W = self.obs_window
        seg = self.net_demand[self.t:self.t + W] / cap
        if len(seg) < W:
            pad = np.full(W - len(seg), seg[-1] if len(seg) else 0.0)
            seg = np.concatenate([seg, pad])
        return seg.astype(np.float32)

    def _lookahead_stats(self):
        cap = max(self.thermal_cap, 1.0)
        L = self.lookahead
        window = self.net_demand[self.t:min(self.t + L, self.horizon)]
        if len(window) < 2:
            return np.zeros(4, dtype=np.float32)
        return np.array([
            float(window.min()) / cap, float(window.max()) / cap,
            float(window.mean()) / cap,
            float(window[-1] - window[0]) / (cap * L),
        ], dtype=np.float32)

    def _obs(self):
        cap = max(self.thermal_cap, 1.0)

        if self.state_version == "v1":
            status_norm = np.clip(self.status_duration / 10.0, -1.0, 1.0)
            return np.concatenate([
                [self.t / max(self.horizon - 1, 1)],
                self.prev_on.astype(float),
                self.net_demand / cap,
                status_norm,
                self._lookahead_stats(),
            ]).astype(np.float32)

        h = self.t % 24
        ang = 2.0 * np.pi * h / 24.0
        remaining = (self.horizon - self.t) / max(self.horizon, 1)
        can_start, can_stop, on_off = self._constraint_clocks()
        return np.concatenate([
            [np.sin(ang), np.cos(ang), remaining],
            self._demand_window(),
            self._lookahead_stats(),
            on_off, can_start, can_stop,
        ]).astype(np.float32)

    def _legalise(self, desired):
        on = np.asarray(desired, int).copy()
        cur_on = (self.current_dispatch > 1e-6) | (self.status_duration > 0)
        on[cur_on & (self.status_duration > 0) & (self.status_duration < self.min_up)] = 1
        on[(~cur_on) & (self.status_duration < 0) & (-self.status_duration < self.min_down)] = 0
        on[cur_on & (on == 0) & (self.current_dispatch > self.ramp_down + 1e-6)] = 1
        return on

    def _update_durations(self, on):
        nxt = np.zeros(self.n_gen, float)
        for i in range(self.n_gen):
            if on[i] == 1:
                nxt[i] = max(1.0, self.status_duration[i] + 1.0) if self.prev_on[i] == 1 else 1.0
            else:
                nxt[i] = min(-1.0, self.status_duration[i] - 1.0) if self.prev_on[i] == 0 else -1.0
        return nxt

    def step(self, action):
        raw = np.asarray(action, int).ravel()[:self.n_gen]
        on  = self._legalise(raw)

        nd_t     = float(self.net_demand[self.t])
        required = max(0.0, nd_t)

        total_pmin_on = float(self.pmin[on == 1].sum())
        total_pmax_on = float(self.pmax[on == 1].sum())

        cur_on = (self.current_dispatch > 1e-6) | (self.status_duration > 0)
        startup_cost = float(np.sum(((on == 1) & (~cur_on)) * self.startup_cost_arr))
        fuel_proxy = float(np.sum(on * self.pmin * self.marginal_cost))
        unserved = max(0.0, required - total_pmax_on)
        over_commit = max(0.0, total_pmin_on - required)

        spill_cost = self.spill_penalty * over_commit

        if required > 0:
            on_indices = np.where(on == 1)[0]
            sorted_by_pmin = on_indices[np.argsort(-self.pmax[on_indices])]
            cumsum = 0.0
            needed = set()
            for idx in sorted_by_pmin:
                if cumsum < required * 1.15:
                    cumsum += self.pmax[idx]
                    needed.add(idx)
            n_excess = len(on_indices) - len(needed)
            excess_unit_penalty = n_excess * 100.0
        else:
            excess_unit_penalty = float(on.sum()) * 500.0

        lookahead_penalty = 0.0
        L = self.lookahead
        t_end = min(self.t + L, self.horizon)
        future_min_demand = float(self.net_demand[self.t:t_end].min())
        if future_min_demand < required * 0.5 and over_commit > 0:
            lookahead_penalty = over_commit * 500.0

        cost = (fuel_proxy + startup_cost + self.balance_penalty * unserved
                + spill_cost + excess_unit_penalty + lookahead_penalty)
        reward = -cost / self.reward_scale

        dur_next = self._update_durations(on)
        self.current_dispatch = on.astype(float) * self.pmin

        info = {
            "instance_id": self.episodes[self.ep_idx]["instance_id"],
            "hour": self.t, "net_demand": nd_t,
            "on": on.copy(), "raw_action": raw.copy(),
            "n_overrides": int(np.sum(on != raw)),
            "startup_cost": startup_cost, "fuel_proxy": fuel_proxy,
            "cost": cost, "reward": reward,
            "unserved_proxy": unserved, "over_commit": over_commit,
            "n_startups": int(((on == 1) & (~cur_on)).sum()),
            "n_online": int(on.sum()),
        }
        self.prev_on = on.copy()
        self.status_duration = dur_next
        self.t += 1
        done = self.t >= self.horizon
        obs  = np.zeros(self.obs_dim, np.float32) if done else self._obs()
        return obs, reward, done, False, info


def _reset(env_obj, idx=None):
    out = env_obj.reset(options={"episode_index": idx}) if idx is not None else env_obj.reset()
    return out[0]

def _step(env_obj, action):
    obs, rew, term, trunc, info = env_obj.step(action)
    return obs, rew, bool(term or trunc), info


env = UCOnlyEnv(train_episodes, spill_penalty=1000, reward_scale=1e5,
                random_reset=True, seed=42, lookahead=8,
                state_version="v2", obs_window=72)
print(f"env ready -- obs_dim={env.obs_dim} | n_gen={env.n_gen} | horizon={env.horizon}")

_obs_check = _reset(env)
assert len(_obs_check) == env.obs_dim
print("Sanity check passed: obs length matches obs_dim.")

test_env = UCOnlyEnv(test_episodes, spill_penalty=1000, reward_scale=1e5,
                      random_reset=False, seed=99, lookahead=8,
                      state_version="v2", obs_window=72)
print(f"test_env ready -- built on {len(test_env.episodes)} test episodes")


## 4. v3 (balance-aware) environment + Gym wrappers

In [ ]:
class UCOnlyEnvV3(UCOnlyEnv):

    def __init__(self, *args, **kwargs):
        kwargs["state_version"] = "v2"
        self._v3_ready = False
        self.starts_so_far = None
        self.prev_spill = 0.0
        self.prev_unserved = 0.0
        super().__init__(*args, **kwargs)

        self._max_starts = np.maximum(
            self.horizon / np.maximum(self.min_up + self.min_down, 1.0), 1.0)
        self._v3_ready = True
        self.state_version = "v3"
        self.obs_dim = 3 + self.obs_window + 4 + 3 * self.n_gen + 4 + self.n_gen
        self.reset(options={"episode_index": 0})

    @property
    def obs_layout(self):
        n, W, o = self.n_gen, self.obs_window, 0
        lay = {}
        for key, size in [("time", 3), ("window", W), ("lookahead", 4),
                          ("on_off", n), ("can_start", n), ("can_stop", n),
                          ("balance", 4), ("starts", n)]:
            lay[key] = (o, o + size); o += size
        return lay

    def reset(self, seed=None, options=None):
        self.starts_so_far = np.zeros(self.n_gen, float)
        self.prev_spill = 0.0
        self.prev_unserved = 0.0
        self.state_version = "v2"
        obs, info = super().reset(seed=seed, options=options)
        if not self._v3_ready:
            return obs, info
        self.state_version = "v3"
        return self._obs(), info

    def _balance_features(self):
        denom = max(float(self.net_demand[self.t]), 1.0)
        on_m = (self.current_dispatch > 1e-6) | (self.status_duration > 0)
        return np.array([
            np.clip(float(self.pmin[on_m].sum()) / denom, 0.0, 3.0),
            np.clip(float(self.pmax[on_m].sum()) / denom, 0.0, 3.0),
            np.clip(self.prev_spill / denom, 0.0, 3.0),
            np.clip(self.prev_unserved / denom, 0.0, 3.0),
        ], dtype=np.float32)

    def _obs(self):
        if not self._v3_ready:
            return super()._obs()
        self.state_version = "v2"
        base = super()._obs()
        self.state_version = "v3"
        return np.concatenate([
            base,
            self._balance_features(),
            np.clip(self.starts_so_far / self._max_starts, 0.0, 1.0),
        ]).astype(np.float32)

    def step(self, action):
        cur_on = (self.current_dispatch > 1e-6) | (self.status_duration > 0)
        self.state_version = "v2"
        obs, rew, done, trunc, info = super().step(action)
        self.state_version = "v3"
        self.starts_so_far = self.starts_so_far + \
            ((info["on"] == 1) & (~cur_on)).astype(float)
        self.prev_spill = info["over_commit"]
        self.prev_unserved = info["unserved_proxy"]
        obs = np.zeros(self.obs_dim, np.float32) if done else self._obs()
        return obs, rew, done, trunc, info


class UCGymWrapper(gym.Env):
    metadata = {"render_modes": []}
    def __init__(self, episodes, seed=42, state_version="v2", obs_window=72, spill_penalty=1000):
        super().__init__()
        self.base = UCOnlyEnv(episodes, spill_penalty=spill_penalty, reward_scale=1e5,
                              random_reset=True, seed=seed, lookahead=8,
                              state_version=state_version, obs_window=obs_window)
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(self.base.obs_dim,), dtype=np.float32)
        self.action_space = spaces.MultiBinary(self.base.n_gen)
    def reset(self, *, seed=None, options=None):
        if seed: super().reset(seed=seed)
        return self.base.reset(options=options)
    def step(self, action): return self.base.step(action)


class UCGymWrapperV3(gym.Env):
    metadata = {"render_modes": []}
    def __init__(self, episodes, seed=42, obs_window=72, spill_penalty=1000):
        super().__init__()
        self.base = UCOnlyEnvV3(episodes, spill_penalty=spill_penalty,
                                reward_scale=1e5, random_reset=True, seed=seed,
                                lookahead=8, obs_window=obs_window)
        self.observation_space = spaces.Box(
            -np.inf, np.inf, shape=(self.base.obs_dim,), dtype=np.float32)
        self.action_space = spaces.MultiBinary(self.base.n_gen)
    def reset(self, *, seed=None, options=None):
        if seed: super().reset(seed=seed)
        return self.base.reset(options=options)
    def step(self, action):
        return self.base.step(action)


_e = UCOnlyEnvV3(train_episodes, spill_penalty=1000, obs_window=72)
_o, _ = _e.reset(options={"episode_index": 0})
assert _o.shape[0] == _e.obs_dim, (_o.shape, _e.obs_dim)
for _ in range(5):
    _o, _r, _d, _, _i = _e.step(np.random.randint(0, 2, _e.n_gen))
print("v3 OK | obs_dim:", _e.obs_dim, "| all finite:", bool(np.all(np.isfinite(_o))))


## 5. Economic dispatch methods

In [ ]:
def _ed_bounds(on, pmin, pmax, current_dispatch, ramp_up, ramp_down, status_duration):
    lower = np.zeros(len(on), float); upper = np.zeros(len(on), float)
    cur_on = (current_dispatch > 1e-6) | (status_duration > 0)
    for i in range(len(on)):
        if on[i] == 0: continue
        if cur_on[i]:
            lower[i] = max(pmin[i], current_dispatch[i] - ramp_down[i])
            upper[i] = min(pmax[i], current_dispatch[i] + ramp_up[i])
        else:
            lower[i] = pmin[i]; upper[i] = min(pmax[i], ramp_up[i])
        if upper[i] < lower[i]: upper[i] = lower[i]
    return lower, upper

def dispatch_merit_order(required_mw, on, pmin, pmax, marginal_cost,
                          current_dispatch, ramp_up, ramp_down, status_duration,
                          levels=ED_LEVELS):
    lower, upper = _ed_bounds(on, pmin, pmax, current_dispatch, ramp_up, ramp_down, status_duration)
    dispatch = np.zeros(len(on), float); remaining = float(required_mw)
    for i in range(len(on)):
        if on[i] == 0: continue
        lo, hi = lower[i], upper[i]; best = lo
        for lv in sorted(levels):
            candidate = lv * pmax[i]
            if candidate < lo - 1e-6 or candidate > hi + 1e-6: continue
            if candidate <= remaining + 1e-6: best = candidate
        if best > 1e-6: best = max(best, pmin[i])
        dispatch[i] = best; remaining -= best
    return dispatch

def dispatch_proportional(required_mw, on, pmin, pmax, marginal_cost,
                           current_dispatch, ramp_up, ramp_down, status_duration,
                           levels=ED_LEVELS):
    lower, upper = _ed_bounds(on, pmin, pmax, current_dispatch, ramp_up, ramp_down, status_duration)
    dispatch = lower.copy(); remaining = max(0.0, float(required_mw) - float(dispatch.sum()))
    headroom = np.maximum(upper - lower, 0.0) * on; total_h = headroom.sum()
    if total_h > 1e-9:
        for i in range(len(on)):
            if on[i] == 0: continue
            frac = headroom[i] / total_h
            dispatch[i] += min(frac * remaining, headroom[i])
    dispatch_snapped = np.zeros(len(on), float)
    for i in range(len(on)):
        if on[i] == 0: continue
        lo, hi = lower[i], upper[i]; target = dispatch[i]
        best_level, best_dist = lo, float("inf")
        for lv in levels:
            candidate = lv * pmax[i]
            if candidate < lo - 1e-6 or candidate > hi + 1e-6: continue
            dist = abs(candidate - target)
            if dist < best_dist: best_dist = dist; best_level = candidate
        if best_level > 1e-6: best_level = max(best_level, pmin[i])
        dispatch_snapped[i] = best_level
    return dispatch_snapped

def dispatch_qp_milp(required_mw, on, pmin, pmax, marginal_cost,
                      current_dispatch, ramp_up, ramp_down, status_duration,
                      levels=ED_LEVELS):
    lower, upper = _ed_bounds(on, pmin, pmax, current_dispatch, ramp_up, ramp_down, status_duration)
    on_idx = np.where(on == 1)[0]; n_on = len(on_idx)
    if n_on == 0: return np.zeros(len(on))
    feasible_mw = []
    for i in on_idx:
        lo, hi = lower[i], upper[i]
        gen_levels = [lv * pmax[i] for lv in levels if lo - 1e-6 <= lv * pmax[i] <= hi + 1e-6]
        if not gen_levels: gen_levels = [lo]
        feasible_mw.append(gen_levels)
    var_list = []
    for i_local, (i_global, mw_options) in enumerate(zip(on_idx, feasible_mw)):
        for mw in mw_options:
            var_list.append((i_local, i_global, mw, marginal_cost[i_global]))
    n_vars = len(var_list)
    c_obj = np.array([v[2] * v[3] for v in var_list])
    rows_pick, cols_pick, data_pick = [], [], []
    for vi, (i_local, i_global, mw, mc) in enumerate(var_list):
        rows_pick.append(i_local); cols_pick.append(vi); data_pick.append(1.0)
    A_pick = csr_matrix((data_pick, (rows_pick, cols_pick)), shape=(n_on, n_vars))
    b_lo_pick = np.ones(n_on); b_hi_pick = np.ones(n_on)
    mw_vals = np.array([v[2] for v in var_list])
    A_demand = csr_matrix(mw_vals.reshape(1, -1))
    b_lo_dem = np.array([max(0.0, float(required_mw))])
    b_hi_dem = np.array([float(pmax[on_idx].sum())])
    A_all = sp_vstack([A_pick, A_demand])
    b_lo_all = np.concatenate([b_lo_pick, b_lo_dem]); b_hi_all = np.concatenate([b_hi_pick, b_hi_dem])
    constraints = LinearConstraint(A_all, b_lo_all, b_hi_all)
    integrality = np.ones(n_vars)
    bounds_milp = Bounds(lb=np.zeros(n_vars), ub=np.ones(n_vars))
    result = milp(c_obj, constraints=constraints, integrality=integrality, bounds=bounds_milp)
    dispatch = np.zeros(len(on), float)
    if result.success:
        for vi, (i_local, i_global, mw, mc) in enumerate(var_list):
            if result.x[vi] > 0.5: dispatch[i_global] = mw
    else:
        dispatch = dispatch_merit_order(required_mw, on, pmin, pmax, marginal_cost,
                                         current_dispatch, ramp_up, ramp_down, status_duration, levels)
    return dispatch

ED_METHODS = {"merit_order": dispatch_merit_order, "proportional": dispatch_proportional,
              "qp_milp": dispatch_qp_milp}
print("ED methods:", list(ED_METHODS.keys()))


def apply_ed_to_uc(uc_env, policy_fn, ed_fn, episodes, instance_index=0, levels=ED_LEVELS):
    obs = _reset(uc_env, instance_index)
    ep = episodes[instance_index]
    n_gen = uc_env.n_gen
    current_dispatch = ep["thermal_df"]["init_power"].to_numpy(float).copy()
    status_duration = ep["thermal_df"]["init_status"].to_numpy(float).copy()
    done, trace = False, []
    while not done:
        a = policy_fn(obs)
        obs, rew_uc, done, info_uc = _step(uc_env, a)
        on = info_uc["on"]
        nd_t = float(ep["net_demand"][info_uc["hour"]])
        required = max(0.0, nd_t)
        dispatch = ed_fn(required_mw=required, on=on, pmin=uc_env.pmin, pmax=uc_env.pmax,
                          marginal_cost=uc_env.marginal_cost, current_dispatch=current_dispatch,
                          ramp_up=uc_env.ramp_up, ramp_down=uc_env.ramp_down,
                          status_duration=status_duration, levels=levels)
        fuel_cost = sum(float(np.interp(dispatch[i], np.asarray(uc_env.cost_curve_mw[i], float),
                       np.asarray(uc_env.cost_curve_usd[i], float)))
                       if dispatch[i] > 1e-9 else 0.0 for i in range(n_gen))
        prev_on_flag = (current_dispatch > 1e-6) | (status_duration > 0)
        startup_cost = float(np.sum(((on == 1) & (~prev_on_flag)) * uc_env.startup_cost_arr))
        thermal_supply = float(dispatch.sum())
        unserved = max(0.0, required - thermal_supply)
        spill = max(0.0, thermal_supply - required) + max(0.0, -nd_t)
        cost = fuel_cost + startup_cost + uc_env.balance_penalty * unserved + 500.0 * spill
        current_dispatch = dispatch.copy()
        new_status = np.zeros(n_gen, float)
        for i in range(n_gen):
            cur = prev_on_flag[i]
            if on[i] == 1:
                new_status[i] = max(1.0, status_duration[i] + 1.0) if cur else 1.0
            else:
                new_status[i] = min(-1.0, status_duration[i] - 1.0) if not cur else -1.0
        status_duration = new_status
        trace.append({"instance_id": ep["instance_id"], "hour": info_uc["hour"],
                       "net_demand": nd_t, "required_mw": required, "on": on.copy(),
                       "dispatch": dispatch.copy(), "fuel_cost": fuel_cost,
                       "startup_cost": startup_cost, "cost": cost, "unserved": unserved,
                       "spill": spill, "n_startups": info_uc["n_startups"],
                       "n_online": int(on.sum()), "feasible": unserved < 1.0})
    return pd.DataFrame(trace)


## 6. Evaluation helpers (O.R. / fulfilment / responsiveness / amplitude)

In [ ]:
def collect_uc_only_trace_test(policy_fn, episodes, instance_index):
    obs = _reset(test_env, instance_index)
    ep = episodes[instance_index]; on_rows = []; done = False
    while not done:
        a = policy_fn(obs)
        obs, rew, done, info = _step(test_env, a)
        on_rows.append(info["on"].copy())
    return np.vstack(on_rows), np.asarray(ep["net_demand"], float)

def responsiveness_for_instance(commitment, net_demand):
    committed_count = commitment.sum(axis=1)
    if np.std(committed_count) <= 1e-9 or np.std(net_demand) <= 1e-9:
        return np.nan
    r, _ = pearsonr(committed_count, net_demand)
    return float(r)

def scheduling_amplitude(commitment):
    return float(np.std(commitment.sum(axis=1)))

def benchmark_cost_for_episode(ep, generator_names, cost_curve_mw, cost_curve_usd, startup_cost_arr):
    gen_to_idx = {g: i for i, g in enumerate(generator_names)}
    oj = ep.get("output_json"); bench_cost = 0.0
    if oj:
        bt = pd.DataFrame(oj.get("Thermal production (MW)", {}))
        bo = pd.DataFrame(oj.get("Is on", {}))
        gc = [c for c in bt.columns if c.lower() != "hour"]
        if gc:
            bdisp = bt[gc].to_numpy(float); bon = bo[gc].to_numpy(float)
            for t in range(len(bdisp)):
                for j, col in enumerate(gc):
                    if col in gen_to_idx:
                        idx = gen_to_idx[col]; mw = bdisp[t, j]
                        if mw > 1e-9:
                            mw_c = np.asarray(cost_curve_mw[idx], float)
                            c_c = np.asarray(cost_curve_usd[idx], float)
                            bench_cost += float(np.interp(mw, mw_c, c_c))
                        if t > 0 and bon[t, j] > 0.5 and bon[t-1, j] < 0.5:
                            bench_cost += startup_cost_arr[idx]
    return bench_cost

def evaluate_policy_raw(policy_fn, n_instances, uc_env_for_cost=None):
    ref = uc_env_for_cost or env
    commitment_rows = []
    for vi in range(n_instances):
        commitment, net_demand = collect_uc_only_trace_test(policy_fn, test_episodes, vi)
        commitment_rows.append({"instance": vi,
            "responsiveness": responsiveness_for_instance(commitment, net_demand),
            "amplitude": scheduling_amplitude(commitment)})
    dispatch_rows = []
    for ed_name, ed_fn in ED_METHODS.items():
        for vi in range(n_instances):
            ep = test_episodes[vi]
            tr = apply_ed_to_uc(test_env, policy_fn, ed_fn, test_episodes, vi)
            rl_cost = tr["fuel_cost"].sum() + tr["startup_cost"].sum()
            bench_cost = benchmark_cost_for_episode(ep, ref.generator_names, ref.cost_curve_mw,
                                                     ref.cost_curve_usd, ref.startup_cost_arr)
            OR = rl_cost / bench_cost if bench_cost > 1 else np.nan
            total_demand = np.maximum(0, tr["net_demand"].to_numpy(float)).sum()
            unserved = tr["unserved"].sum()
            fulfilment_pct = max(0, 1 - unserved / max(total_demand, 1e-6)) * 100
            dispatch_rows.append({"instance": vi, "ed_method": ed_name, "OR": OR,
                                   "fulfilment_pct": fulfilment_pct,
                                   "unserved_mwh": unserved, "spill_mwh": tr["spill"].sum()})
    return pd.DataFrame(commitment_rows), pd.DataFrame(dispatch_rows)

print("Evaluation helpers ready.")


## 7. Seed loop -- train the 2x2 factorial 3x each

In [ ]:
from stable_baselines3 import PPO as SB3_PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

SEEDS = [42, 43, 44]
N_EVAL_INSTANCES = 100
SAVEDIR = Path("seed_replication_2x2"); SAVEDIR.mkdir(exist_ok=True)

TOTAL_TS = 300 * 72 * 5
PPO_KW_BASE = dict(n_steps=256, batch_size=64, gamma=0.99, gae_lambda=0.95,
                    learning_rate=3e-4, ent_coef=0.01, clip_range=0.2,
                    n_epochs=10, verbose=0, device=str(DEVICE))

class StatsCB(BaseCallback):
    def __init__(self, tag="", every=200):
        super().__init__(); self.tag, self.every = tag, every; self.n_ep = 0
    def _on_step(self):
        if bool(self.locals["dones"][0]):
            self.n_ep += 1
            if self.n_ep % self.every == 0:
                print(f"    [{self.tag}] episode {self.n_ep}")
        return True

CONFIGS = {
    "v2_sp1000": dict(wrapper=UCGymWrapper,   eval_env_cls="v2", state_version="v2", spill_penalty=1000),
    "v3_sp1000": dict(wrapper=UCGymWrapperV3, eval_env_cls="v3", state_version=None, spill_penalty=1000),
    "v2_sp3000": dict(wrapper=UCGymWrapper,   eval_env_cls="v2", state_version="v2", spill_penalty=3000),
    "v3_sp3000": dict(wrapper=UCGymWrapperV3, eval_env_cls="v3", state_version=None, spill_penalty=3000),
}

def make_train_env(cfg, seed):
    kw = dict(episodes=train_episodes, seed=seed, obs_window=72, spill_penalty=cfg["spill_penalty"])
    if cfg["state_version"] is not None:
        kw["state_version"] = cfg["state_version"]
    return cfg["wrapper"](**kw)

def make_eval_env(cfg):
    if cfg["eval_env_cls"] == "v3":
        return UCOnlyEnvV3(test_episodes, spill_penalty=cfg["spill_penalty"],
                            random_reset=False, obs_window=72)
    return UCOnlyEnv(test_episodes, spill_penalty=cfg["spill_penalty"],
                      random_reset=False, state_version="v2", obs_window=72)

all_seed_rows = []
t0_all = time.time()

for cfg_name, cfg in CONFIGS.items():
    for seed in SEEDS:
        tag = f"{cfg_name}_seed{seed}"
        print(f"\n=== Training {tag} ({time.time()-t0_all:.0f}s elapsed) ===")
        venv = DummyVecEnv([lambda cfg=cfg, seed=seed: Monitor(make_train_env(cfg, seed))])
        model = SB3_PPO("MlpPolicy", venv, seed=seed, **PPO_KW_BASE)
        model.learn(total_timesteps=TOTAL_TS, callback=StatsCB(tag=tag))
        model.save(SAVEDIR / f"ppo_{tag}")

        eval_env_for_policy = make_eval_env(cfg)
        eval_env_for_cost   = env
        policy_fn = lambda o, m=model: np.asarray(m.predict(o, deterministic=True)[0]).ravel()

        global test_env
        _old_test_env = test_env
        test_env = eval_env_for_policy
        commit_df, disp_df = evaluate_policy_raw(policy_fn, N_EVAL_INSTANCES, uc_env_for_cost=eval_env_for_cost)
        test_env = _old_test_env

        commit_df.to_csv(SAVEDIR / f"{tag}_commitment.csv", index=False)
        disp_df.to_csv(SAVEDIR / f"{tag}_dispatch.csv", index=False)

        prop = disp_df[disp_df.ed_method == "proportional"]
        row = {
            "config": cfg_name, "seed": seed,
            "responsiveness": commit_df["responsiveness"].mean(skipna=True),
            "amplitude": commit_df["amplitude"].mean(),
            "OR_median_proportional": prop["OR"].median(),
            "fulfilment_pct_proportional": prop["fulfilment_pct"].mean(),
            "spill_mwh_proportional": prop["spill_mwh"].mean(),
            "unserved_mwh_proportional": prop["unserved_mwh"].mean(),
        }
        all_seed_rows.append(row)
        print(f"  {tag}: responsiveness={row['responsiveness']:.3f} | "
              f"O.R.={row['OR_median_proportional']:.3f} | "
              f"fulfilment={row['fulfilment_pct_proportional']:.1f}%")

seed_summary = pd.DataFrame(all_seed_rows)
seed_summary.to_csv(SAVEDIR / "seed_replication_2x2_raw.csv", index=False)
print(f"\nAll done in {(time.time()-t0_all)/3600:.1f}h. Raw per-seed results saved.")


## 8. Aggregate across seeds -- mean +/- std per configuration

In [ ]:
agg = seed_summary.groupby("config").agg(
    responsiveness_mean=("responsiveness", "mean"), responsiveness_std=("responsiveness", "std"),
    amplitude_mean=("amplitude", "mean"), amplitude_std=("amplitude", "std"),
    OR_mean=("OR_median_proportional", "mean"), OR_std=("OR_median_proportional", "std"),
    fulfilment_mean=("fulfilment_pct_proportional", "mean"), fulfilment_std=("fulfilment_pct_proportional", "std"),
    spill_mean=("spill_mwh_proportional", "mean"), spill_std=("spill_mwh_proportional", "std"),
).round(3)

agg.to_csv(SAVEDIR / "seed_replication_2x2_summary.csv")
print(agg.to_string())
print("\nCompare these means to the single-run 2x2 numbers already in the paper draft")
print("(Table 4: v2/1000=0.000, v3/1000=0.295, v2/3000=0.275, v3/3000=0.829).")
print("If the std for responsiveness is small relative to the gaps between configs,")
print("the interaction claim survives seed replication.")


## 9. Bonus -- finer k-grid for the static-fleet sweep (40-45)

In [ ]:
K_VALUES_FINE = [10, 15, 20, 25, 30, 35, 40, 41, 42, 43, 44, 45]

def static_commitment_mask_k(env_obj, k):
    order = np.argsort(env_obj.marginal_cost)
    idx = np.sort(order[:k])
    mask = np.zeros(env_obj.n_gen, dtype=float)
    mask[idx] = 1.0
    return mask

def apply_ed_to_static(ed_fn, ep, gen_mask, levels=ED_LEVELS):
    n_gen = env.n_gen
    current_dispatch = ep["thermal_df"]["init_power"].to_numpy(float).copy() * gen_mask
    status_duration   = ep["thermal_df"]["init_status"].to_numpy(float).copy()
    status_duration   = np.where(gen_mask > 0, status_duration, -1.0)
    trace = []
    for t in range(len(ep["net_demand"])):
        on = gen_mask.copy()
        nd_t = float(ep["net_demand"][t])
        required = max(0.0, nd_t)
        dispatch = ed_fn(required_mw=required, on=on, pmin=env.pmin, pmax=env.pmax,
                          marginal_cost=env.marginal_cost, current_dispatch=current_dispatch,
                          ramp_up=env.ramp_up, ramp_down=env.ramp_down,
                          status_duration=status_duration, levels=levels)
        fuel_cost = sum(float(np.interp(dispatch[i], np.asarray(env.cost_curve_mw[i], float),
                       np.asarray(env.cost_curve_usd[i], float)))
                       if dispatch[i] > 1e-9 else 0.0 for i in range(n_gen))
        prev_on_flag = (current_dispatch > 1e-6) | (status_duration > 0)
        startup_cost = float(np.sum(((on == 1) & (~prev_on_flag)) * env.startup_cost_arr))
        thermal_supply = float(dispatch.sum())
        unserved = max(0.0, required - thermal_supply)
        spill    = max(0.0, thermal_supply - required) + max(0.0, -nd_t)
        cost     = fuel_cost + startup_cost + ep["balance_penalty"] * unserved + 500.0 * spill
        current_dispatch = dispatch.copy()
        new_status = np.zeros(n_gen, float)
        for i in range(n_gen):
            cur = prev_on_flag[i]
            if on[i] == 1:
                new_status[i] = max(1.0, status_duration[i] + 1.0) if cur else 1.0
            else:
                new_status[i] = min(-1.0, status_duration[i] - 1.0) if not cur else -1.0
        status_duration = new_status
        trace.append({"hour": t, "net_demand": nd_t, "fuel_cost": fuel_cost,
                       "startup_cost": startup_cost, "cost": cost,
                       "unserved": unserved, "spill": spill, "n_online": int(on.sum())})
    return pd.DataFrame(trace)


static_fine_rows = []
t0_static = time.time()
for k in K_VALUES_FINE:
    mask = static_commitment_mask_k(env, k)
    cap = env.pmax[mask > 0].sum()
    for ed_name, ed_fn in ED_METHODS.items():
        ORs, unserved_l, spill_l, total_demand_l = [], [], [], []
        for vi in range(N_TEST_INSTANCES_EVAL if "N_TEST_INSTANCES_EVAL" in dir() else 100):
            ep = test_episodes[vi]
            tr = apply_ed_to_static(ed_fn, ep, mask)
            rl_cost = tr["fuel_cost"].sum() + tr["startup_cost"].sum()
            bench_cost = benchmark_cost_for_episode(ep, env.generator_names, env.cost_curve_mw,
                                                     env.cost_curve_usd, env.startup_cost_arr)
            ORs.append(rl_cost / bench_cost if bench_cost > 1 else np.nan)
            unserved_l.append(tr["unserved"].sum())
            spill_l.append(tr["spill"].sum())
            total_demand_l.append(float(np.maximum(0, tr["net_demand"].to_numpy(float)).sum()))
        ORs = np.asarray(ORs, dtype=float)
        unserved_a = np.asarray(unserved_l); total_demand_a = np.asarray(total_demand_l)
        fulfilment_per_instance = np.maximum(0, 1 - unserved_a / np.maximum(total_demand_a, 1e-6)) * 100
        static_fine_rows.append({
            "k": k, "ed_method": ed_name,
            "or_p25": np.nanpercentile(ORs, 25), "or_p50": np.nanpercentile(ORs, 50),
            "or_p75": np.nanpercentile(ORs, 75),
            "fulfilment_pct": float(np.mean(fulfilment_per_instance)),
            "avg_spill_mwh": float(np.mean(spill_l)), "avg_unserved_mwh": float(np.mean(unserved_l)),
            "capacity_mw": cap,
        })
    print(f"  k={k:2d} done ({time.time()-t0_static:.0f}s elapsed)")

static_fine_df = pd.DataFrame(static_fine_rows)
static_fine_df.to_csv("static_fleet_sweep_fine_40_45.csv", index=False)
print(f"\nSaved static_fleet_sweep_fine_40_45.csv ({len(static_fine_df)} rows)")
display(static_fine_df.round(3))
